# PROSIT 2 — Autonomous Driving Perception
### All Sprints Orchestrator Notebook

This notebook orchestrates the existing `src/` scripts from the repository. It clones the repo, installs dependencies, then runs each sprint end-to-end.

> **Before running**: Push the project to GitHub, then set `REPO_URL` in Cell 1.

## 0. Environment Setup

In [ ]:
# ── 0.1  Clone repo from GitHub and install dependencies ────────────────────
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # <── set this
REPO_DIR = "/content/prosit2"

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(f"{REPO_DIR}/prosits/object-detection")
print(f"Working dir: {os.getcwd()}")

!pip install -q albumentations scipy google-generativeai tqdm opencv-python-headless
print("✅ Dependencies installed.")

In [ ]:
# ── 0.2  Mount Google Drive and configure paths ──────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import torch

DRIVE_ROOT  = Path("/content/drive/MyDrive/prosit2")
IDD_TRAIN   = DRIVE_ROOT / "data/idd_yolo/train"
IDD_VAL     = DRIVE_ROOT / "data/idd_yolo/val"
GHANA_DIR   = DRIVE_ROOT / "data/raw/ghana"
CKPT_DIR    = DRIVE_ROOT / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = DRIVE_ROOT / "results";     RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device      : {DEVICE}")
print(f"IDD train   : {IDD_TRAIN}  (exists: {IDD_TRAIN.exists()})")
print(f"IDD val     : {IDD_VAL}    (exists: {IDD_VAL.exists()})")
print(f"Ghana frames: {GHANA_DIR}  (exists: {GHANA_DIR.exists()})")

---
## Sprint 1 — Road Segmentation & Domain Augmentation Ablation

**Goal**: Train a U-Net segmentation model on clean CamVid data (Experiment A), then re-train with Ghanaian-condition augmentations (Experiment B). The ablation study proves augmentation dramatically improves zero-shot generalization on Ghanaian footage.

In [ ]:
# ── 1.1  (OPTIONAL) Extract frames from Ghana dashcam videos ─────────────────
# Only needed if you haven't already extracted frames to Drive
# !python src/data/frame_extractor.py \
#     --video_dir {DRIVE_ROOT}/videos \
#     --output_dir {GHANA_DIR} \
#     --fps 3
print("Skipping frame extraction — frames already on Drive.")

In [ ]:
# ── 1.2  (OPTIONAL) Auto-annotate Ghana frames with classical CV pseudo-masks ─
# Only needed if you don't have ground-truth masks for Sprint 1 evaluation
# !python src/data/auto_annotator.py \
#     --input_dir  {GHANA_DIR} \
#     --output_dir {DRIVE_ROOT}/data/annotations/classical
print("Skipping annotation — using existing masks.")

In [ ]:
# ── 1.3  Train: Experiment A — Baseline (NO augmentation) ───────────────────
!python src/train.py \
    --data_dir       {DRIVE_ROOT}/data/seg \
    --checkpoint_dir {CKPT_DIR} \
    --epochs         20 \
    --batch_size     8

In [ ]:
# ── 1.4  Train: Experiment B — Augmented (domain-specific augmentations) ─────
!python src/train_augmented.py \
    --data_dir       {DRIVE_ROOT}/data/seg \
    --checkpoint_dir {CKPT_DIR} \
    --epochs         20 \
    --batch_size     8

In [ ]:
# ── 1.5  Ablation: compare both models on Ghanaian footage ───────────────────
!python src/ablation.py \
    --baseline_model  {CKPT_DIR}/baseline_best.pth \
    --augmented_model {CKPT_DIR}/augmented_best.pth \
    --ghana_data_dir  {GHANA_DIR} \
    --ghana_mask_dir  {DRIVE_ROOT}/data/annotations/classical \
    --output_dir      {RESULTS_DIR}/ablation/

In [ ]:
# ── 1.6  Print Sprint 1 ablation summary ────────────────────────────────────
summary_path = f"{RESULTS_DIR}/ablation/summary.txt"
if Path(summary_path).exists():
    print(open(summary_path).read())
else:
    print("summary.txt not found — check output_dir above.")

---
## Sprint 2 — Custom Object Detection

**Goal**: Build and train a grid-based single-stage detector from scratch on IDD bounding box data.

**Key constraints**: All loss functions (Objectness + Localisation + Classification) and NMS are implemented in pure PyTorch — no YOLO library shortcuts.

In [ ]:
# ── 2.1  (OPTIONAL) Convert IDD → YOLO format ────────────────────────────────
# Only needed once — skips if already converted
if not IDD_TRAIN.exists():
    IDD_RAW = DRIVE_ROOT / "data/IDD_Detection_Organized"
    !python src/data/convert_idd.py \
        --idd_dir    {IDD_RAW} \
        --output_dir {DRIVE_ROOT}/data/idd_yolo
else:
    print(f"IDD YOLO dataset already exists ({IDD_TRAIN}) — skipping conversion.")

In [ ]:
# ── 2.2  Train: Custom detector (U-Net encoder + grid prediction head) ────────
!python src/train_detector.py \
    --data_dir       {IDD_TRAIN} \
    --checkpoint_dir {CKPT_DIR} \
    --epochs         30 \
    --batch_size     4 \
    --img_size       416

In [ ]:
# ── 2.3  Evaluate: mAP@0.5 + JSONL event log with distance estimates ─────────
!python src/evaluate_detector.py \
    --data_dir    {IDD_VAL} \
    --model_path  {CKPT_DIR}/detector_best.pth \
    --output_dir  {RESULTS_DIR}/sprint2/ \
    --img_size    416 \
    --num_save    20

In [ ]:
# ── 2.4  Sprint 2 results summary ───────────────────────────────────────────
metrics_path = f"{RESULTS_DIR}/sprint2/metrics.txt"
logfile_path = f"{RESULTS_DIR}/sprint2/event_log.jsonl"

print("=== Detection Metrics ===")
print(open(metrics_path).read() if Path(metrics_path).exists() else "metrics.txt not found")

import json
if Path(logfile_path).exists():
    events = [json.loads(l) for l in open(logfile_path)]
    print(f"\n=== Event Log ===")
    print(f"Total detections with distance estimates: {len(events)}")
    print("\nSample entries:")
    for e in events[:5]:
        print(f"  [{e['class']:12s}] conf={e['confidence']:.2f}  "
              f"dist={e['distance_x_m']}m  offset={e['offset_y_m']}m")

---
## Sprint 3 — Self-Aware Failure Mining

**Goal**: Use the Gemini VLM to automatically diagnose why the Sprint 2 detector fails on real Ghanaian footage. Mine hard-negative frames, then retrain with failure-mode-targeted augmentations.

**Deliverable**: Show mAP improvement from Sprint 2 → Sprint 3.

In [ ]:
# ── 3.1  Run VLM failure mining ─────────────────────────────────────────────
import os
GEMINI_KEY = "YOUR_KEY_HERE"    # <── paste your Gemini API key here
os.environ["GEMINI_API_KEY"] = GEMINI_KEY

!python src/vlm/failure_miner.py \
    --ghana_dir    {GHANA_DIR} \
    --model_path   {CKPT_DIR}/detector_best.pth \
    --api_key      {GEMINI_KEY} \
    --output_dir   {RESULTS_DIR}/failure_mining/ \
    --max_failures 50 \
    --img_size     416

In [ ]:
# ── 3.2  Inspect failure mode distribution ──────────────────────────────────
import json
from collections import Counter

fa_path = f"{RESULTS_DIR}/failure_mining/failure_analysis.json"
if Path(fa_path).exists():
    results = json.load(open(fa_path))
    modes   = Counter(r['vlm_analysis'].get('dominant_failure_mode','unknown') for r in results)
    total   = sum(modes.values())
    print(f"Failure frames analyzed: {len(results)}\n")
    print(f"{'Failure Mode':<22}  Count  Pct")
    print("-" * 38)
    for m, c in modes.most_common():
        print(f"  {m:<20}  {c:4d}   {c/total*100:.1f}%")
else:
    print("failure_analysis.json not found — run cell 3.1 first.")

In [ ]:
# ── 3.3  Retrain with failure-mode targeted augmentations ────────────────────
!python src/train_detector_v2.py \
    --data_dir         {IDD_TRAIN} \
    --pretrained       {CKPT_DIR}/detector_best.pth \
    --failure_analysis {RESULTS_DIR}/failure_mining/failure_analysis.json \
    --checkpoint_dir   {CKPT_DIR} \
    --epochs           20 \
    --batch_size       4 \
    --lr               5e-5

In [ ]:
# ── 3.4  Evaluate Sprint 2 vs Sprint 3 mAP@0.5 ──────────────────────────────
print("Evaluating Sprint 2 (baseline)...")
!python src/evaluate_detector.py \
    --data_dir    {IDD_VAL} \
    --model_path  {CKPT_DIR}/detector_best.pth \
    --output_dir  {RESULTS_DIR}/sprint3_v1/ \
    --img_size    416 \
    --num_save    0

print("\nEvaluating Sprint 3 (retrained with hard negatives)...")
!python src/evaluate_detector.py \
    --data_dir    {IDD_VAL} \
    --model_path  {CKPT_DIR}/detector_v2_best.pth \
    --output_dir  {RESULTS_DIR}/sprint3_v2/ \
    --img_size    416 \
    --num_save    0

In [ ]:
# ── 3.5  Final ablation comparison ──────────────────────────────────────────
def read_map_from_metrics(path):
    p = Path(path) / "metrics.txt"
    if not p.exists(): return None
    for line in open(p):
        if line.startswith("mAP@0.5:"):
            return float(line.split(":")[1].strip().rstrip("%"))
    return None

map_v1 = read_map_from_metrics(f"{RESULTS_DIR}/sprint3_v1")
map_v2 = read_map_from_metrics(f"{RESULTS_DIR}/sprint3_v2")

print("=" * 45)
print("          Sprint Ablation Results")
print("=" * 45)
if map_v1 is not None:
    print(f"  Sprint 2 (baseline):   mAP@0.5 = {map_v1:.2f}%")
if map_v2 is not None:
    print(f"  Sprint 3 (retrained):  mAP@0.5 = {map_v2:.2f}%")
if map_v1 and map_v2:
    delta = map_v2 - map_v1
    print(f"  Improvement:                   {delta:+.2f}%")
print("=" * 45)